# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [1]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping


DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  11.0 #/s median value from BRENDA
DEFAULT_KCAT_TRANSPORT = 100000000 #/s

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_ecolik12_240726.xlsx')

GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_eco_230309.json')

/bin/bash: line 1: fg: no job control
/bin/bash: line 1: fg: no job control
/bin/bash: line 1: fg: no job control
Loading PAModelpy modules version 0.0.4.1
Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iML1515_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'{formatted_date}_eco_ActiveEnzymeInformation_GotEnzymes.xlsx')

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'\b([b|s]\d{4})\b'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BiGG database](http://bigg.ucsd.edu/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iML1515 model for [*Escherichia coli* K12 substr. MG1655](http://bigg.ucsd.edu/models/iML1515)
- Important: the identifiers must me mappable to uniprot (or another enzyme identifier database) and KEGG (for mapping to GotEnzymes)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `eco` (*E. coli* strain K12) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *E. coli* we used [this](https://www.uniprot.org/uniprotkb?query=%28organism_id%3A83333%29) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [4]:
#load the model
model = read_sbml_model(os.path.join('Models', 'iML1515.xml'))
expand=False

#make a id mapping and create a mapping dataframe
id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'rhea_id', 'mnx_id','Reactants','Products','EC', 'GPR'])
for rxn in model.reactions:
    #skip transport reactions and ATP maintenance requirement
    if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
        continue
    to_append= []   
    to_append.append(rxn.annotation['bigg.reaction'])
    #not all reactions are annotated with a KEGG ID in the model
    try: 
        to_append.append(rxn.annotation['kegg.reaction'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['rhea'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['metanetx.reaction'])
    except:
        to_append.append(np.nan)
    try:
        to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
    except:
        to_append.append([])
    try: 
        to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
    except:
        to_append.append([])
    if 'ec-code' in rxn.annotation.keys():
        #if there are multiple enzymes make seperate lists
        if isinstance(rxn.annotation['ec-code'], list) and expand:
            info = to_append.copy()
            for i in range(len(rxn.annotation['ec-code'])):
                #make a new row for each enzyme by copying the previous information
                info = to_append.copy()
                ec= rxn.annotation['ec-code'][i]
                info.append(ec)
                #add gpr
                info.append(rxn.gene_reaction_rule)
                #add to df
                id_mapper_df.loc[len(id_mapper_df)] = info
        else:
            to_append.append(rxn.annotation['ec-code'])
            to_append.append(rxn.gene_reaction_rule)
            id_mapper_df.loc[len(id_mapper_df)] = to_append
    else:
        to_append.append(np.nan)
        #ignore entries without enzyme and gpr relation
        if rxn.gene_reaction_rule =='':
            continue
        else:
            to_append.append(rxn.gene_reaction_rule)
        id_mapper_df.loc[len(id_mapper_df)] = to_append
        
#remove entries without GRP relation or Ec number


print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
print('Number of reactions mapped to rhea identifier: ', len(id_mapper_df.rhea_id.dropna()))
print('Number of reactions mapped to MetaNetX identifier: ', len(id_mapper_df.mnx_id.dropna()))
id_mapper_df.head()

Number of reactions in the mapping dataframe:  1944
Number of reactions mapped to KEGG identifier:  741
Number of reactions mapped to rhea identifier:  970
Number of reactions mapped to MetaNetX identifier:  1823


,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125
3,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518
4,SHK3Dr,R02413,"[17739, 17738, 17737, 17740]",MNXR104378,"[C02637, C00080, C00005]","[C00006, C00493]","[1.1.1.25, 1.1.1.282]",b3281 or b1692


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 2.1 Parse the BIGG gene-protein-reaction associations

In [5]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    return re.findall(locustag_regex, text)

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066,b2066
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238,b0238
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0238
3,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0125
4,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518,b0474
...,...,...,...,...,...,...,...,...,...
3527,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857
3528,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856
3529,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857
3530,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001


### 2.2 Parse the Uniprot data for merging

In [6]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
uniprot_df

/home/samiralvdb/Software/anaconda3/envs/PAModelpy/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Entry,Entry Name,Length,Mass,Rhea ID,b_number
0,A5A616,MGTS_ECOLI,31,3509,NaN,b4599
1,O32583,THIS_ECOLI,66,7311,NaN,b4407
2,P00350,6PGD_ECOLI,468,51481,RHEA:10116,b2029
3,P00363,FRDA_ECOLI,602,65972,RHEA:40523 RHEA:27834,b4154
4,P00370,DHE4_ECOLI,447,48581,RHEA:11612,b1761
...,...,...,...,...,...,...
4582,Q9S4X4,YUBB_ECOLI,145,16507,NaN,NaN
4583,Q9S4X5,YUBA_ECOLI,168,19067,NaN,NaN
4584,Q9XB42,YKFH_ECOLI,73,8581,NaN,b4504
4585,Q9Z3A0,YJGW_ECOLI,111,13085,NaN,b4274


### 2.3 Match Uniprot to BIGG data

In [7]:
id_mapper_with_protein = pd.merge(id_mapper_df, uniprot_df, on='b_number', how='left')

id_mapper_with_protein

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,Entry,Entry Name,Length,Mass,Rhea ID
0,CYTDK2,R00517,"[28165, 28164, 28162, 28163]",MNXR97042,"[C00475, C00044]","[C00055, C00035, C00080]","[2.7.1.48, 2.7.1.213]",b2066,b2066,P0A8F4,URK_ECOLI,213.0,24353.0,RHEA:16825 RHEA:24674
1,XPPT,R02142,"[10800, 10803, 10802, 10801]",MNXR105243,"[C00119, C00385]","[C00013, C00655]","[2.4.2.8, 2.4.2.22]",b0238,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
2,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0238,P0A9M5,XGPT_ECOLI,152.0,16971.0,RHEA:25424 RHEA:10800 RHEA:17973
3,HXPRT,R01132,"[17976, 17974, 17973, 17975]",MNXR100752,"[C00262, C00119]","[C00130, C00013]",2.4.2.8,b0238 or b0125,b0125,P0A9M2,HPRT_ECOLI,178.0,20115.0,RHEA:17973 RHEA:25424
4,NDPK5,R01857,"[27692, 27690, 27691, 27693]",MNXR96118,"[C00002, C00361]","[C00008, C00286]",2.7.4.6,b0474 or b2518,b0474,P69441,KAD_ECOLI,214.0,23586.0,RHEA:12973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7267,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
7268,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3856,P32125,MOBB_ECOLI,175.0,19363.0,NaN
7269,BMOGDS2,NaN,NaN,MNXR96316,[],[],NaN,(b3857 and b3856) or b3857,b3857,P32173,MOBA_ECOLI,194.0,21643.0,RHEA:34243
7270,FESD2s,NaN,NaN,MNXR99586,[],[],NaN,s0001,s0001,NaN,NaN,NaN,NaN,NaN


## 3. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [8]:
id_mapper_final = id_mapper_with_protein
id_mapper_final = id_mapper_final.rename({'Entry':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

KeyError: "['rhea_id_up'] not found in axis"

In [ ]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

In [ ]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [ ]:
# Get directionalities and fill in the gaps
eco_enzymes_merged['direction'] = 'f'
for index, row in eco_enzymes_merged.iterrows():
    #there is a kegg id and kcat associated to this reaction
    if not isinstance(row.kcat_values, float):
        if row.compound in row.Products:
            eco_enzymes_merged.direction.iloc[index] = 'b'
        elif not row.compound in row.Reactants:
            print(f'Compound {row["compound"]} is not associated to reaction {row["kegg_id"]}')
            continue
    #there is no kcat associated to this reaction, try to use EC number to get a kcat value
    else: 
        got_enzyme_info = eco_enzymes_df[eco_enzymes_df['ec_number']==row.EC]
        for i, info in got_enzyme_info.iterrows():
            #is there any enzyme association if we look at the metabolites in the reaction and the EC number?
            if info.compound in row.Products:
                direction = 'b'
            elif info.compound in row.Reactants:
                direction = 'f'
            else:
                continue
            #change the kcat if it is not there already or create a new enzyme association
            if np.isnan(row.kcat_values):
                eco_enzymes_merged.direction.iloc[index] = 'b'
                eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
            else:
                eco_enzymes_merged.loc[len(eco_enzymes_merged)] = row.to_list()[:-2]+[info.kcat_values, 'b']
        #add default information for unmappable proteins
        if np.isnan(row.kcat_values) & (isinstance(row.enzyme_id, str)) & (row.GPR!= 's0001'):
            eco_enzymes_merged.direction.iloc[index] = 'f'
            eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
        if np.isnan(row.Mass) & (len(row.GPR)>2) & (row.GPR!= 's0001'):
            eco_enzymes_merged.Mass.iloc[index] = DEFAULT_PROT_MW
        if row.GPR == 's0001':
            eco_enzymes_merged.gene.iloc[index] = [row.GPR]

        
eco_enzymes_mapped = eco_enzymes_merged.drop(['ec_number', 'compound'], axis=1).rename({'Mass':'molMass'}, axis =1)
eco_enzymes_mapped

In [ ]:
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

In [ ]:
#save the dataframe
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(subset = ['rxn_id', 'enzyme_id', 'direction'],
                                                       ignore_index=True)
eco_enzymes_mapped.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [ ]:
# Get the information about the enzyme sectors
unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'ExcessEnzymes')
translational_protein_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'Translational')

In [ ]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_mapped.to_excel(writer, sheet_name='ActiveEnzymes')
    translational_protein_df.to_excel(writer, sheet_name='Translational')
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme')